In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import zipfile
tf.__version__

**Path do dataset e criação do gerador de imagens**


In [ ]:
path_dataset = r'/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset'
image_data_get =tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function = tf.keras.applications.resnet50.preprocess_input,
    validation_split = 0.2
)


**Carregamento das imagens a partir do path e divisão entre treino e validação**

In [ ]:
image_gen_train = image_data_get.flow_from_directory(
    path_dataset,
    target_size = (224,224),
    batch_size = 32,
    class_mode = 'categorical',
    shuffle = True,
    subset = 'training'
)
image_gen_val = image_data_get.flow_from_directory(
    path_dataset,
    target_size = (224,224),
    batch_size = 32,
    class_mode = 'categorical',
    shuffle = True,
    subset = 'validation'
)

**Verificando índices**

In [ ]:
image_gen_train.class_indices

**Carregando modelo de transferência a partir de arquivo local**

In [ ]:
modelo_transfer = r'/kaggle/input/transfer-leraning/keras/default/1/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5'
transfer_model = tf.keras.applications.ResNet50(
    weights=modelo_transfer,
    include_top=False,
    input_shape=(224,224,3)
)
display("Done")

**Testando qualidade da imagem**

In [ ]:
testing_image = r'/kaggle/input/augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset/ModerateDemented/001ef5a6-893b-4ede-9cf8-60b7fb94a541.jpg'

plt.imshow(plt.imread(testing_image))

**Verificando camadas entre convolucionais e densas**

In [ ]:
for i, layers in enumerate(transfer_model.layers):
  print(i, layers)

**Salvando camadas de output**

In [ ]:
x = transfer_model.output

**Adicionando camadas densas de outputs**

In [ ]:
x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dense(1024, activation='relu')(x)
x = tf.keras.layers.Dense(512, activation='relu')(x)
preds = tf.keras.layers.Dense(4, activation='softmax')(x)

**Mesclando modelo de transferência com novo output**

In [ ]:
model = tf.keras.models.Model(inputs=transfer_model.input, outputs=preds)

**Verificando sumário da rede**

In [ ]:
model.summary()

**Verificando imagem pós-processamento**

In [ ]:
imagem, label = image_gen_train[1][0][0], image_gen_train[1][1][0]
plt.imshow(imagem)
plt.title(f"Class: {label}")
plt.axis('off')


In [ ]:
image_gen_train[1][0][0]

**Compilando e treinando modelo**

In [ ]:
model.compile(optimizer=tf.keras.optimizers.RMSprop(0.0001, decay=1e-6), loss='categorical_crossentropy',metrics=['accuracy'])
hist = model.fit(image_gen_train, validation_data=image_gen_train, epochs = 10, batch_size=512)

**Verificando acurácia e sua validação durante o treino**

In [ ]:
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']

In [ ]:
plt.plot(acc, label='Accuracy')
plt.plot(val_acc, label="Accuracy Validation")
plt.xlabel("Epochs")
plt.ylabel("Accuracy/Validation")
plt.figure()

**Verificando perda e sua validação durante o treino**

In [ ]:
loss = hist.history['loss']
val_loss = hist.history['val_loss']

In [ ]:
plt.plot(loss, label='Loss')
plt.plot(val_loss, label="Loss Validation")
plt.xlabel("Epochs")
plt.ylabel("Loss/Validation")
plt.figure()

**Path de testes**

In [ ]:
path_testing_dataset = r'/kaggle/input/augmented-alzheimer-mri-dataset/OriginalDataset'

**Criando gerador de imagens teste e processando-as**

In [ ]:
testing_image_data_gen = tf.keras.preprocessing.image.ImageDataGenerator(  
   preprocessing_function = tf.keras.applications.resnet50.preprocess_input
)

images = testing_image_data_gen.flow_from_directory(
    path_testing_dataset,
    target_size = (224,224),
    shuffle = False
)

**Verificando imagens da base de teste**

In [ ]:
l = 10
w = 10

fig, axes = plt.subplots(l, w, figsize = (15,15))
axes = axes.ravel()

for i in np.arange(0, l*w):
    random_index = random.randint(0, 200)
    axes[i].imshow(images[random_index][0][0])
    axes[i].set_title(f"Class: {np.argmax(images[random_index][1][0])}")
    axes[i].axis('off')
plt.subplots_adjust(hspace=1)

**Prevendo classes**

In [ ]:

pred = model.predict(images)
pred_classes = np.argmax(pred, axis=1)


**Verificando acurácia, matriz de confusão, e metricas de classificação do modelo com a base de teste**

In [ ]:
true_classes = images.classes
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report
accuracy_score(true_classes, pred_classes)

In [ ]:
cm = confusion_matrix(true_classes, pred_classes)
sns.heatmap(cm, annot=True)

In [ ]:
cr = classification_report(true_classes, pred_classes, target_names=images.class_indices.keys())
print(cr)

**Visualizando comparação entre classe real e previsão do modelo**

In [ ]:
l = 10
w = 10

fig, axes = plt.subplots(l, w, figsize = (15,15))
axes = axes.ravel()

true_classes = images.classes
class_labels = list(images.class_indices.keys())

for i in np.arange(0, l*w):
    random_index = random.randint(0, len(pred_classes)-1)
    img_path = images.filepaths[random_index]
    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    img = tf.keras.utils.img_to_array(img) / 255.0  

    axes[i].imshow(img)
    axes[i].set_title(f"True: {true_classes[random_index]}\n"
                      f"Pred: {pred_classes[random_index]}")
    axes[i].axis('off')
plt.subplots_adjust(hspace=.7)